# RQ2: evaluated state checks

This notebook recomputes the selected 14 synthetic checks and 12 checks across ten physical actions. P03 uses the recorded hardware retest; the original pilot is retained as source evidence for the remaining actions and contextual snapshots.

Saved tables and results are included below; no rerun is needed to inspect them. Regeneration is optional.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                  if (p / "notebooks").is_dir() and (p / "src" / "indoortwin" / "paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
import json
import numpy as np
import pandas as pd
from IPython.display import display
DATA = REPO_ROOT / "data/rq2"
OUT = REPO_ROOT / "results/rq2_states"
(OUT / "tables").mkdir(parents=True, exist_ok=True)
synth = pd.read_csv(DATA / "EXP_RQ2_SYNTH_20261001_055355/rq2_synthetic_matrix_results.csv")
checks = pd.read_csv(DATA / "EXP_RQ2_CONSOLIDATED/rq2_consolidated_results.csv")
metrics = json.loads((DATA / "EXP_RQ2_CONSOLIDATED/rq2_chapter5_metrics.json").read_text())
integration = json.loads((DATA / "EXP_RQ2_LIVE_V11_20261001_185959/integration_summary.json").read_text())
def flags(values):
    return values.astype(str).str.lower().eq("true")
assert len(synth) == 14 and flags(synth["pass"]).all()
assert len(checks) == 12 and flags(checks["pass"]).all()
assert checks["expected"].eq(checks["observed"]).all()
assert checks["step_id"].nunique() == 10
assert integration["physical_properties"] == 27 and integration["derived_properties"] == 10
rows = []
for step, group in checks.groupby("step_id", sort=True):
    rows.append({"step_id": step, "step_name": group["step_name"].iloc[0],
                 "expected_state": " + ".join(group["expected"].astype(str)),
                 "observed_state": " + ".join(group["observed"].astype(str)),
                 "checks": len(group), "checks_passed": int(flags(group["pass"]).sum()),
                 "latency_ms": int(group["latency_ms"].max()), "pass": bool(flags(group["pass"]).all()),
                 "experiment_id": group["evidence"].iloc[0]})
steps = pd.DataFrame(rows)
assert steps.loc[steps.step_id.str.endswith("P03"), "latency_ms"].iloc[0] == 2018
assert checks.latency_ms.min() == metrics["latency_ms"]["min"]
assert checks.latency_ms.max() == metrics["latency_ms"]["max"]
checks.to_csv(OUT / "tables/rq2_expected_state_checks.csv", index=False)
steps.to_csv(OUT / "tables/table_5_2_state_transition_timing.csv", index=False)
synth.to_csv(OUT / "tables/rq2_synthetic_checks.csv", index=False)
summary = {"synthetic_checks_passed": int(flags(synth["pass"]).sum()), "physical_checks_passed": int(flags(checks["pass"]).sum()),
           "physical_actions": len(steps), "latency_min_ms": int(checks.latency_ms.min()),
           "latency_max_ms": int(checks.latency_ms.max()), "physical_properties": integration["physical_properties"],
           "calculated_states": integration["derived_properties"]}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(steps)
display(summary)


,step_id,step_name,expected_state,observed_state,checks,checks_passed,latency_ms,pass,experiment_id
0,RQ2-P01,Unoccupied fans off,UNOCCUPIED + VENTILATION_OFF,UNOCCUPIED + VENTILATION_OFF,2,2,2,True,EXP_RQ2_PHYS_V11_20261001_190449
1,RQ2-P02,Occupant enters,OCCUPIED,OCCUPIED,1,1,1,True,EXP_RQ2_PHYS_V11_20261001_190449
2,RQ2-P03,All fans run,VENTILATION_ALL_RUNNING,VENTILATION_ALL_RUNNING,1,1,2018,True,EXP_RQ2_P03_RETEST_20261001_212343
3,RQ2-P04,Exhaust commanded no rotation,COMMAND_NO_ROTATION,COMMAND_NO_ROTATION,1,1,3,True,EXP_RQ2_PHYS_V11_20261001_190449
4,RQ2-P05,Exhaust restored,FAN_RESPONSE_NORMAL,FAN_RESPONSE_NORMAL,1,1,4,True,EXP_RQ2_PHYS_V11_20261001_190449
5,RQ2-P06,Window open,OPENING_PRESENT,OPENING_PRESENT,1,1,1,True,EXP_RQ2_PHYS_V11_20261001_190449
6,RQ2-P07,Window closed,ALL_CLOSED,ALL_CLOSED,1,1,2,True,EXP_RQ2_PHYS_V11_20261001_190449
7,RQ2-P08,Heater level 1,HEATER_LEVEL1 + OCCUPIED_HEATING_CLOSED_ROOM,HEATER_LEVEL1 + OCCUPIED_HEATING_CLOSED_ROOM,2,2,1,True,EXP_RQ2_PHYS_V11_20261001_190449
8,RQ2-P09,Heater off,HEATER_OFF,HEATER_OFF,1,1,2,True,EXP_RQ2_PHYS_V11_20261001_190449
9,RQ2-P10,Occupant leaves,UNOCCUPIED,UNOCCUPIED,1,1,3,True,EXP_RQ2_PHYS_V11_20261001_190449


{'synthetic_checks_passed': 14,
 'physical_checks_passed': 12,
 'physical_actions': 10,
 'latency_min_ms': 1,
 'latency_max_ms': 2018,
 'physical_properties': 27,
 'calculated_states': 10}

These are host-observed state-transition times from a consolidated physical sequence. The P03 retest is not an additional independent replicate. Later matched-start comparisons and the 1 October CO₂ trajectory are outside the current draft's reported RQ2 results and are excluded.

## Synthetic state checks

In [2]:
display(synth)


,scenario_id,name,entity_property,expected,observed,pass,reason
0,RQ2-01,"Occupied, normal CO2",ROOM01.occupancy_context,OCCUPIED,OCCUPIED,True,Derived from authoritative occupancy=PRESENT.
1,RQ2-01,"Occupied, normal CO2",ROOM01.co2_state,CO2_NORMAL,CO2_NORMAL,True,Indoor CO2=800.0 ppm
2,RQ2-02,"Occupied, high CO2",ROOM01.safety_context,OCCUPIED_HIGH_CO2,OCCUPIED_HIGH_CO2,True,Occupied room with high CO2.
3,RQ2-03,Occupancy stale,ROOM01.occupancy_context,OCCUPANCY_UNKNOWN,OCCUPANCY_UNKNOWN,True,Occupancy input unavailable or stale.
4,RQ2-04,All fans off,ROOM01.ventilation_state,VENTILATION_OFF,VENTILATION_OFF,True,No fan is commanded or physically running.
5,RQ2-05,All fans running,ROOM01.ventilation_state,VENTILATION_ALL_RUNNING,VENTILATION_ALL_RUNNING,True,All three fans have RPM above the development ...
6,RQ2-06,One or two fans running,ROOM01.ventilation_state,VENTILATION_PARTIAL,VENTILATION_PARTIAL,True,1/3 fans physically running.
7,RQ2-07,Command but no RPM,FAN-EXH.health_context,COMMAND_NO_ROTATION,COMMAND_NO_ROTATION,True,Fan commanded above zero but RPM below running...
8,RQ2-08,Rotation with zero command,FAN-EXH.health_context,UNCOMMANDED_ROTATION,UNCOMMANDED_ROTATION,True,RPM indicates rotation while desired PWM is zero.
9,RQ2-09,Window/door open,ROOM01.opening_context,OPENING_PRESENT,OPENING_PRESENT,True,NaN


## Individual physical state checks

In [3]:
display(checks)


,step_id,step_name,entity_property,expected,observed,pass,latency_ms,evidence
0,RQ2-P01,Unoccupied fans off,ROOM01.occupancy_context,UNOCCUPIED,UNOCCUPIED,True,2,EXP_RQ2_PHYS_V11_20261001_190449
1,RQ2-P01,Unoccupied fans off,ROOM01.ventilation_state,VENTILATION_OFF,VENTILATION_OFF,True,2,EXP_RQ2_PHYS_V11_20261001_190449
2,RQ2-P02,Occupant enters,ROOM01.occupancy_context,OCCUPIED,OCCUPIED,True,1,EXP_RQ2_PHYS_V11_20261001_190449
3,RQ2-P03,All fans run,ROOM01.ventilation_state,VENTILATION_ALL_RUNNING,VENTILATION_ALL_RUNNING,True,2018,EXP_RQ2_P03_RETEST_20261001_212343
4,RQ2-P04,Exhaust commanded no rotation,FAN-EXH.health_context,COMMAND_NO_ROTATION,COMMAND_NO_ROTATION,True,3,EXP_RQ2_PHYS_V11_20261001_190449
5,RQ2-P05,Exhaust restored,FAN-EXH.health_context,FAN_RESPONSE_NORMAL,FAN_RESPONSE_NORMAL,True,4,EXP_RQ2_PHYS_V11_20261001_190449
6,RQ2-P06,Window open,ROOM01.opening_context,OPENING_PRESENT,OPENING_PRESENT,True,1,EXP_RQ2_PHYS_V11_20261001_190449
7,RQ2-P07,Window closed,ROOM01.opening_context,ALL_CLOSED,ALL_CLOSED,True,2,EXP_RQ2_PHYS_V11_20261001_190449
8,RQ2-P08,Heater level 1,ROOM01.heater_context,HEATER_LEVEL1,HEATER_LEVEL1,True,1,EXP_RQ2_PHYS_V11_20261001_190449
9,RQ2-P08,Heater level 1,ROOM01.safety_context,OCCUPIED_HEATING_CLOSED_ROOM,OCCUPIED_HEATING_CLOSED_ROOM,True,1,EXP_RQ2_PHYS_V11_20261001_190449
